# Notebook 02 — Preprocessing

Pipeline: lowercase → noise removal → tokenise → remove html-junk tokens → POS tag → lemmatise → stopword removal → flag empties → store

In [1]:
import re
import pickle
import pandas as pd
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

DATASET_PATH = "../data/raw/Emotion_classify_Data.csv"
df = pd.read_csv(DATASET_PATH)

## 2.0 Drop contradictory duplicates

EDA (Notebook 01) identified 3 comment pairs that appear under two different labels — always `anger` vs `fear`. All 6 rows are dropped before the pipeline runs to prevent any of them leaking across the train/val/test split.

In [2]:
conflict_comments = (
    df.groupby("Comment")["Emotion"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)
df = df[~df["Comment"].isin(conflict_comments)].reset_index(drop=True)
print(f"Rows after dropping contradictory duplicates: {len(df)}")

Rows after dropping contradictory duplicates: 5931


**Result:** 5,931 rows (6 dropped — the 3 contradictory pairs).

## 2.1 Lowercase + noise removal

Lowercase before stripping so all patterns are in a consistent case.
The regex keeps only `[a-zA-Z\s]` — punctuation, digits, and html special characters are removed. Result is stored in `Cleaned_Comment`; `Comment` is preserved untouched.

In [3]:
df['Comment'] = df['Comment'].str.lower()
df["Cleaned_Comment"] = df["Comment"].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))
df["Cleaned_Comment"]


0       i seriously hate one subject to death but now ...
1                      im so full of life i feel appalled
2       i sit here to write i start to dig out my feel...
3       ive been really angry with r and i feel like a...
4       i feel suspicious if there is no one outside l...
                              ...                        
5926                   i begun to feel distressed for you
5927    i left feeling annoyed and angry thinking that...
5928    i were to ever get married i d have everything...
5929    i feel reluctant in applying there because i w...
5930    i just wanted to apologize to you because i fe...
Name: Cleaned_Comment, Length: 5931, dtype: str

**Result:** Lowercased; non-alphabetic characters stripped. Alphabetic html artifacts (`www`, `href`, `src`, `img`, etc.) survive this step — removed post-tokenisation in 2.3.

## 2.2 Tokenisation

Split `Cleaned_Comment` into a list of word tokens using `nltk.word_tokenize()`. Output is stored in a new `tokens` column — all downstream steps (POS tagging, lemmatisation, stopword removal) operate on this list, never the raw string.

`word_tokenize` is preferred over `.split()` because it handles contractions: `"don't"` → `["do", "n't"]`, `"i'm"` → `["i", "'m"]`. 

In [4]:
df['tokens'] = df['Cleaned_Comment'].apply(lambda x: nltk.tokenize.word_tokenize(x, language='english', preserve_line=False))
df['tokens'].head(5)

0    [i, seriously, hate, one, subject, to, death, ...
1          [im, so, full, of, life, i, feel, appalled]
2    [i, sit, here, to, write, i, start, to, dig, o...
3    [ive, been, really, angry, with, r, and, i, fe...
4    [i, feel, suspicious, if, there, is, no, one, ...
Name: tokens, dtype: object

## 2.3 Remove html-junk tokens

After tokenisation, filter out any token that is a known html artifact — fragments like `www`, `href`, `src`, `img`, `rel`, `bookmark`, `class` that survive the regex strip because they are purely alphabetic.

Filtering at the token level (rather than with a broader regex pre-tokenisation) keeps the change surgical: only exact matches on a fixed list are removed, with no risk of clobbering real words with similar substrings.

In [5]:
HTML_JUNK = {
    "www", "http", "https", "href", "src", "img",
    "rel", "bookmark", "class", "aligncenter",
    "clairee", "bondmusings",
}

df["Cleaned_Tokens"] = df["tokens"].apply(lambda toks: [t for t in toks if t not in HTML_JUNK])
df["Cleaned_Tokens"].head(5)

0    [i, seriously, hate, one, subject, to, death, ...
1          [im, so, full, of, life, i, feel, appalled]
2    [i, sit, here, to, write, i, start, to, dig, o...
3    [ive, been, really, angry, with, r, and, i, fe...
4    [i, feel, suspicious, if, there, is, no, one, ...
Name: Cleaned_Tokens, dtype: object

**Result:**

## 2.4 POS tagging

POS tagging happens **before** stopword removal — the tagger uses auxiliary verbs and articles as context clues to correctly label surrounding words.
Removing stopwords first degrades tagging accuracy, which in turn degrades lemmatisation.

> `nltk.pos_tag()` returns Penn Treebank tags (`VBG`, `NN`, `JJ`...). These need to be mapped to WordNet's simpler format before lemmatisation — that mapping is the next step.

In [6]:
df['POS_Tags'] = df['Cleaned_Tokens'].apply(lambda x: nltk.pos_tag(x))
df['POS_Tags'].head(5)

0    [(i, NN), (seriously, RB), (hate, VB), (one, C...
1    [(im, NNS), (so, RB), (full, JJ), (of, IN), (l...
2    [(i, JJ), (sit, NN), (here, RB), (to, TO), (wr...
3    [(ive, JJ), (been, VBN), (really, RB), (angry,...
4    [(i, NN), (feel, VBP), (suspicious, JJ), (if, ...
Name: POS_Tags, dtype: object

In [7]:
print(f'Number of unique tokens: {len(df["POS_Tags"].explode().unique())}')

Number of unique tokens: 11559


**Result:**

## 2.5 Lemmatisation

`WordNetLemmatizer` requires WordNet POS tags (`v`, `n`, `a`, `r`), not Penn Treebank tags.
Write a `get_wordnet_pos(tag)` helper that maps the first character of the Penn tag to the WordNet equivalent — default to `n` (noun) for anything that doesn't match.
Then lemmatise each token using its mapped tag.

> This is tag-aware lemmatisation: `"feeling"` as a verb → `"feel"`, as a noun → `"feeling"`. Blind lemmatisation (no tag) defaults everything to noun and gets verbs wrong.

**Known limitation:** `WordNetLemmatizer` correctly collapses regular verb forms (`"feeling"` → `"feel"`) but fails on irregular forms — `"felt"` is left unchanged even when verb-tagged. This is a documented gap in WordNet's lemmatisation dictionary, verified by isolated testing (`lemmatizer.lemmatize("felt", wordnet.VERB)` returns `"felt"`). Irregular forms like `"felt"`, `"was"`, `"went"` will survive as distinct tokens rather than collapsing to their base form.

In [8]:
def get_wordnet_pos(pos_tag):
    if pos_tag.startswith('J'):
        return wordnet.ADJ
    elif pos_tag.startswith('V'):
        return wordnet.VERB
    elif pos_tag.startswith('N'):
        return wordnet.NOUN
    elif pos_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

lemmatizer = WordNetLemmatizer()

**Result:** Helper and lemmatizer defined. `.apply()` runs in 2.5b after sample verification.

### 2.5a Before/after verification

Before running `.apply()` on all 5,931 rows — confirm the logic works correctly on a small independent test. If there's a bug in `get_wordnet_pos` (a wrong mapping, a missed case) this is where you catch it cheaply, before it silently corrupts the full column.

The spaCy comparison below also runs here: it confirms the irregular-verb limitation (`felt`, `went`, `had`, `was`) is specific to WordNet's dictionary, not a bug in the pipeline logic.

In [9]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("i went to school and had an argument with my friend. I was very angry and frustrated.")
for token in doc:
    print(token.text, token.lemma_)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/fionnadavoodian/.pyenv/versions/3.11.9/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/fionnadavoodian/.pyenv/versions/3.11.9/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/fionnadavoodian/.pyenv/versions/3.11.9/lib/python3.11/site-packages/ipykernel/kernelapp

i I
went go
to to
school school
and and
had have
an an
argument argument
with with
my my
friend friend
. .
I I
was be
very very
angry angry
and and
frustrated frustrate
. .


**Result:** spaCy comparison confirmed — irregular verb forms (`felt`, `went`, `had`, `was`) are a WordNet dictionary gap, not a logic bug. Logic verified on a small sample; scaling to full dataset in 2.5b.

### 2.5b Run on full dataset

Sample verified — apply lemmatisation to all 5,931 rows, then confirm the vocabulary reduction and spot-check the key word variants (`feeling`, `feels`, `felt`) across the full column.

In [10]:
df['Lemmatized_Tokens'] = df['POS_Tags'].apply(
    lambda tagged: [lemmatizer.lemmatize(tok, get_wordnet_pos(tag)) for tok, tag in tagged]
)

before = len(df['Cleaned_Tokens'].explode().unique())
after = len(df['Lemmatized_Tokens'].explode().unique())
print(f"Unique tokens before lemmatization: {before:,}")
print(f"Unique tokens after lemmatization:  {after:,}")
print(f"Vocabulary reduction: {(before - after) / before:.0%}")

Unique tokens before lemmatization: 8,960
Unique tokens after lemmatization:  7,312
Vocabulary reduction: 18%


In [11]:
target_words = ["feeling", "feels", "felt"]

mask = df['Cleaned_Tokens'].apply(lambda tokens: any(target_word in tokens for target_word in target_words))

matches = df[mask]
print(f"Rows containing '{target_words}': {len(matches)}")

for idx in matches.index[:20]:
    original_tokens = df.loc[idx, 'Cleaned_Tokens']       
    lemmatized_tokens = df.loc[idx, 'Lemmatized_Tokens']
    
    if "feeling" in original_tokens :
        position = original_tokens.index("feeling")
    elif "feels" in original_tokens :
        position = original_tokens.index("feels")
    elif "felt" in original_tokens :
        position = original_tokens.index("felt") 
    else :
        None
    
    print(f"Row {idx}:")
    print(f"  original word:   {original_tokens[position]}")
    print(f"  became:          {lemmatized_tokens[position]}")

Rows containing '['feeling', 'feels', 'felt']': 2068
Row 7:
  original word:   feeling
  became:          feel
Row 11:
  original word:   feeling
  became:          feel
Row 13:
  original word:   feeling
  became:          feel
Row 18:
  original word:   feeling
  became:          feel
Row 20:
  original word:   feeling
  became:          feeling
Row 22:
  original word:   feeling
  became:          feel
Row 24:
  original word:   feeling
  became:          feeling
Row 25:
  original word:   feeling
  became:          feeling
Row 27:
  original word:   feeling
  became:          feeling
Row 28:
  original word:   feeling
  became:          feel
Row 34:
  original word:   feeling
  became:          feel
Row 36:
  original word:   feeling
  became:          feel
Row 38:
  original word:   felt
  became:          felt
Row 39:
  original word:   felt
  became:          felt
Row 40:
  original word:   feeling
  became:          feel
Row 41:
  original word:   feeling
  became:          fee

In [12]:
before = len(df['POS_Tags'].explode().unique())
after = len(df['Lemmatized_Tokens'].explode().unique())
print(f"Unique tokens before lemmatization: {before}")
print(f"Unique tokens after lemmatization: {after}")

Unique tokens before lemmatization: 11559
Unique tokens after lemmatization: 7312


**Result:** Lemmatisation applied to all 5,931 rows. Vocabulary reduced by 18% (8,960 → 7,312 unique tokens). Word check on 2,068 rows: `feeling` → `feel` when verb-tagged, stays `feeling` when noun-tagged (both correct). `felt` → `felt` — WordNet gap confirmed.

## 2.6 Stopword removal

Remove common low-signal words **after** lemmatisation using a plain Python set — not `TfidfVectorizer` or any automated vectoriser.
Assignment requires features to be built from tokens manually; no vectoriser should touch the pipeline here.

> Use `sklearn.feature_extraction.text.ENGLISH_STOP_WORDS` or `nltk.corpus.stopwords.words('english')` as your set.
> Explicitly remove negations (`not`, `no`, `nor`, `never`) from the stopword set before filtering — they carry direct emotion signal.

In [16]:
negation_words = {"not", "no", "nor", "never", "none", "nothing", "neither", "nowhere", "hardly", "scarcely", "barely"}
stop_words = set(stopwords.words('english')) - negation_words

df["filtered_tokens"] = df['Lemmatized_Tokens'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

mask = df['Lemmatized_Tokens'].apply(lambda tokens: any(word in tokens for word in negation_words))

matches = df[mask]
print(f"Rows containing a negation word: {len(matches)}")

for idx in matches.index[:5]:
    original = df.loc[idx, 'Lemmatized_Tokens']
    filtered = df.loc[idx, 'filtered_tokens']

    print(f"Row {idx}:")
    print(f"  original:  {original}")
    print(f"  filtered:  {filtered}")

Rows containing a negation word: 932
Row 2:
  original:  ['i', 'sit', 'here', 'to', 'write', 'i', 'start', 'to', 'dig', 'out', 'my', 'feeling', 'and', 'i', 'think', 'that', 'i', 'be', 'afraid', 'to', 'accept', 'the', 'possibility', 'that', 'he', 'might', 'not', 'make', 'it']
  filtered:  ['sit', 'write', 'start', 'dig', 'feeling', 'think', 'afraid', 'accept', 'possibility', 'might', 'not', 'make']
Row 4:
  original:  ['i', 'feel', 'suspicious', 'if', 'there', 'be', 'no', 'one', 'outside', 'like', 'the', 'rapture', 'have', 'happen', 'or', 'something']
  filtered:  ['feel', 'suspicious', 'no', 'one', 'outside', 'like', 'rapture', 'happen', 'something']
Row 10:
  original:  ['i', 'feel', 'a', 'bit', 'like', 'franz', 'liebkind', 'in', 'the', 'producer', 'not', 'many', 'people', 'know', 'it', 'but', 'the', 'fuhrer', 'be', 'a', 'terrific', 'dancer']
  filtered:  ['feel', 'bit', 'like', 'franz', 'liebkind', 'producer', 'not', 'many', 'people', 'know', 'fuhrer', 'terrific', 'dancer']
Row 13:
 

**Result:** 932 rows contain at least one preserved negation. Row 13 — *"i do not always find myself feel thankful"* — filtered to `['not', 'always', 'find', 'feel', 'thankful', ...]`. Without `not`, those tokens are indistinguishable from a genuinely thankful sentence: `find`, `feel`, `thankful`, `grateful`, `thanks` all present with no signal the sentence is negated. The person is expressing the *absence* of thankfulness — keeping `not` is what makes that classifiable.

## 2.7 Flag empty / near-empty rows

After all transformations, some comments may have 0 or 1 token left.
Flag rather than silently drop — print the original `Comment` for any flagged rows so the decision to remove them is deliberate and documented.

In [ ]:
df['filtered_tokens'] = df['filtered_tokens'].apply(lambda tokens: [word for word in tokens if len(word) > 1])

5931

**Result:**

## 2.8 Store tokens column

Store the final lemmatised, filtered token lists as a `tokens` column in the dataframe.
This is the output all downstream steps (feature engineering, modelling) will consume.

TypeError: 'tuple' object is not callable

**Result:**

## 2.9 Label encoding

Convert string labels to integers. Save the encoder to preserve the class mapping for decoding predictions at inference time.

> Print the mapping explicitly: `dict(zip(le.classes_, le.transform(le.classes_)))`

**Result:**

## 2.10 Train / validation / test split

Stratified 70 / 15 / 15 split to preserve class proportions in all three sets.

**Result:**

## 2.11 Verify split balance

Confirm class ratios held across all three splits — a failed stratification silently skews everything downstream.

**Result:**

## 2.12 Save

Persist the processed dataframe and label encoder so Notebook 03 (feature engineering) loads clean, split data without re-running this pipeline.

**Result:**